<a href="https://colab.research.google.com/github/rija-ansari/MSE1003H_RijaAnsari/blob/main/Assignment_3/MSE1003_Assignment3_RA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Comment out the below cells when running in Colab

In [ ]:
#!git clone https://github.com/rija-ansari/MSE1003H_RijaAnsari/
#!git clone https://github.com/rija-ansari/MSE1003H_RijaAnsari/tree/main/Assignment_3

In [ ]:
#%cd Assignment_3

# Assignment 3: Active Learning for Single-Objective Optimization

## Introduction

### Overview
In this assignment, the goal is to design an active learning strategy that efficiently discovers a red–yellow–blue (RYB) mixture whose 8-channel optical response matches a given target response measured on the Opentron-2 (OT-2) platform. Each experiment corresponds to a choice of dye volumes (R, Y, B) with a fixed total volume of 300 µL, and the OT-2 returns an 8-dimensional intensity vector across different wavelength channels.

The input is as follows:

$x = (R, Y, B), \quad R + Y + B = 300\ \mu\text{L}$

and the corresponding 8-channel output is:


$f(x) = (ch583, ch670, ch510, ch410, ch620, ch470, ch550, ch440).$


The target output for this assignment is:

$y^{\text{target}} = (6794, 7313, 4227, 554, 9325, 2635, 5758, 2061)$

### Single-objective optimization

To make this a single-objective optimization, the scalar objective is defined as the difference between OT-2 output response and the target response. The Euclidean distance is used after normalization: 

$J(x) = \left\|\tilde{f}(x) - \tilde{y}^{\text{target}}\right\|_2^2,$


where $\tilde{f}(x)$ and $\tilde{y}^{\text{target}}$ are the channel-wise normalized response and target, respectively. 

The optimization task is then simply:

$\min_{x \in \mathcal{X}} J(x), \quad \text{subject to } R + Y + B = 300,\ R, Y, B \ge 10$



Because direct evaluation of $J(x)$ requires running an experiment on the OT-2, we will train a surrogate model to approximate the mapping $x \mapsto J(x)$ using existing data (from Assignment 2 and additional student data). 



## Analysis

### Set up environment and assignment folder

Create a new assignment folder and navigate to it
- cd /Users/rija/MSE1003H_RijaAnsari/Assignment_3

Create a virtual environment in the terminal 
- python -m venv .venv  

Create a new text file with the name ".gitignore"
- add the text venv/,pycache/ and .env (if used)

Issues arrived from multiple python versions that kept conflicting with each other  

**Before beginning ensure that Python 3.13.9 is running**

Check everything is in order:
- Move back to the main repo
    - cd /Users/rija/MSE1003H_RijaAnsari
- Make sure this is the main repository url
    - git remote -v
- We need to add our new files from the assignment folder
    - git add . 
    - git commit -m "Assignment 3 structure update"
    - git pull origin main
    - git push origin main

### Data Import and Cleaning

In [ ]:
pip install gpytorch

In [ ]:
import os
import ast
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from plotly import express as px

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_squared_error, root_mean_squared_error, r2_score, mean_absolute_error, max_error, mean_absolute_percentage_error

from sklearn.gaussian_process import GaussianProcessRegressor as GPR
from sklearn.multioutput import MultiOutputRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, Matern, RationalQuadratic, ConstantKernel


import torch
import gpytorch
from gpytorch.means import ConstantMean, MultitaskMean
from gpytorch.kernels import RBFKernel, ScaleKernel, IndexKernel, MultitaskKernel
from gpytorch.distributions import MultitaskMultivariateNormal


random_seed = 1003

In [ ]:
#target definition

target = pd.Series(
    {
        "ch583": 6794,
        "ch670": 7313,
        "ch510": 4227,
        "ch410": 554,
        "ch620": 9325,
        "ch470": 2635,
        "ch550": 5758,
        "ch440": 2061,
    },
    name="target",
)

target


We need to compile previous students data to train the surrogate model. I will be conserving my Assignment 2 data as a test set:

- **Students_data.zip:**  RYB–response pairs collected by other students that will be the primary training data.
- **Background_filtered_data.csv (optional):** Background noise measurements due to ambient light at different positions. When used, these measurements can be subtracted from raw responses to obtain background-corrected intensities.
- **Assignment 2 data:** RYB input and 8-channel output for test.


In [ ]:
#read single csv files

data_dir = Path(os.getcwd()) 

a2_path = data_dir / "color_results.csv"
background_path = data_dir / "background_filtered_data.csv"


# Load what is available; wrap in try/except so the notebook is robust
def safe_read_csv(path, label):
    if path.exists():
        print(f"Loading {label} from {path}")
        return pd.read_csv(path)
    else:
        print(f"WARNING: {label} file not found at {path}. Skipping.")
        return None

a2_df = safe_read_csv(a2_path, "Assignment 2 data")
background_df = safe_read_csv(background_path, "Background data")

a2_df.head() if a2_df is not None else None


In [ ]:
#compile students data into one dataframe

students_folder = Path(data_dir) / "students_data"  

# Find all CSV files in the folder
csv_files = sorted(students_folder.glob("*.csv"))

if not csv_files:
    raise FileNotFoundError(f"No CSV files found in: {students_folder}")

print(f"Found {len(csv_files)} CSV files in {students_folder}")


compiled_dfs = []

for file in csv_files:
    try:
        df_i = pd.read_csv(file)
        df_i["source"] = file.stem  # tag each row with the filename (without extension)
        compiled_dfs.append(df_i)
        print(f"Loaded {file.name} — shape {df_i.shape}")
    except Exception as e:
        print(f"⚠️ Skipping {file.name} due to read error: {e}")

# Concatenate all loaded CSVs
students_df = pd.concat(compiled_dfs, ignore_index=True)

print("\nFinal merged dataframe shape:", students_df.shape)
students_df.head()


In [ ]:
# Convert stringified dicts into real dicts
students_df["Command"] = students_df["Command"].apply(ast.literal_eval)
students_df["Sensor Data"] = students_df["Sensor Data"].apply(ast.literal_eval)


In [ ]:
# Expand Command (RYB volumes)
cmd_expanded = pd.json_normalize(students_df["Command"])
cmd_expanded = cmd_expanded.rename(columns={"R": "R", "Y": "Y", "B": "B"})

# Expand Sensor Data (8 channels)
sensor_expanded = pd.json_normalize(students_df["Sensor Data"])


In [ ]:
flat_df = pd.concat(
    [
        cmd_expanded,          # R, Y, B
        sensor_expanded,       # ch583 ... ch440
        students_df[["Experiment ID", "timestamp", "source"]]
    ],
    axis=1
)

print("Flattened dataframe shape:", flat_df.shape)
flat_df.head()


In [ ]:
# Identify channel columns automatically
channel_cols = [c for c in flat_df.columns if c.startswith("ch")]

clean_df = flat_df[["R", "Y", "B"] + channel_cols].copy()

print("Clean dataframe shape:", clean_df.shape)
clean_df.head()


### Exploratory Data Analysis

Before training the surrogate model, it is important to understand:

- The coverage of the RYB design space  
- The distribution of raw channel intensities  
- Correlations between channels  
- Whether the dataset contains clusters, gaps, or outliers  

Let's split the data before we get started so we don't have any leakage

In [ ]:
"""#train test split of dataframe

df = clean_df.copy()

#X is the volumes of RYB and Y is the 8 channels
X = df[["R", "Y", "B"]]
y = df[[f"{ch}" for ch in channel_cols]]

# Perform train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=random_seed)

print("Training set size:", X_train.shape, y_train.shape)
print("Test set size:", X_test.shape, y_test.shape)
"""
print("Results from simple train/test split are saved in 'simple_split_results.csv'")

In [ ]:
"""df = clean_df.copy()

# X is the volumes of RYB and y is the 8 channels
X = df[["R", "Y", "B"]].to_numpy()
y = df[[f"{ch}" for ch in channel_cols]].to_numpy()

# 1. Bin each dimension into quantiles
R_bins = pd.qcut(X[:, 0], q=3, labels=False, duplicates='drop')
Y_bins = pd.qcut(X[:, 1], q=3, labels=False, duplicates='drop')
B_bins = pd.qcut(X[:, 2], q=3, labels=False, duplicates='drop')

# 2. Combine bins into a single stratification label
strata = R_bins * 100 + Y_bins * 10 + B_bins

# 3. Perform a stratified split
from sklearn.model_selection import StratifiedShuffleSplit

splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

for train_idx, val_idx in splitter.split(X, strata):
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]
"""

In [ ]:
"""df = clean_df.copy()

# X is the volumes of RYB and y is the 8 channels
X = df[["R", "Y", "B"]]                      # <-- DataFrame
y = df[[f"{ch}" for ch in channel_cols]]     # <-- DataFrame

# 1. Bin each dimension into quantiles (3 bins to avoid sparse strata)
R_bins = pd.qcut(X["R"], q=3, labels=False, duplicates='drop')
Y_bins = pd.qcut(X["Y"], q=3, labels=False, duplicates='drop')
B_bins = pd.qcut(X["B"], q=3, labels=False, duplicates='drop')

# 2. Combine bins into a single stratification label
strata = R_bins * 100 + Y_bins * 10 + B_bins

# 3. Perform a stratified split
from sklearn.model_selection import StratifiedShuffleSplit

splitter = StratifiedShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

for train_idx, val_idx in splitter.split(X, strata):
    X_train = X.iloc[train_idx].copy()
    X_val   = X.iloc[val_idx].copy()
    y_train = y.iloc[train_idx].copy()
    y_val   = y.iloc[val_idx].copy()"""


In [ ]:
"""df = clean_df.copy().sort_values(by=["R", "Y", "B"]).reset_index(drop=True)

# Use 3 bins (27 strata) — stable and avoids sparse bins
R_bins = pd.qcut(df["R"], q=[0, 1/3, 2/3, 1], labels=False)
Y_bins = pd.qcut(df["Y"], q=[0, 1/3, 2/3, 1], labels=False)
B_bins = pd.qcut(df["B"], q=[0, 1/3, 2/3, 1], labels=False)

strata = R_bins * 100 + Y_bins * 10 + B_bins

splitter = StratifiedShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=random_seed)

X = df[["R", "Y", "B"]]
y = df[[f"{ch}" for ch in channel_cols]]

for train_idx, val_idx in splitter.split(X, strata):
    X_train = X.iloc[train_idx].copy()
    X_val   = X.iloc[val_idx].copy()
    y_train = y.iloc[train_idx].copy()
    y_val   = y.iloc[val_idx].copy()

"""

In [ ]:
"""from sklearn.cluster import KMeans
from sklearn.model_selection import StratifiedShuffleSplit

df = clean_df.copy().sort_values(by=["R", "Y", "B"]).reset_index(drop=True)

X = df[["R", "Y", "B"]]
y = df[[f"{ch}" for ch in channel_cols]]

# 1. Create spatial clusters
kmeans = KMeans(n_clusters=8, random_state=123)
clusters = kmeans.fit_predict(X)

# 2. Stratified split using cluster labels
splitter = StratifiedShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=random_seed
)

for train_idx, val_idx in splitter.split(X, clusters):
    X_train = X.iloc[train_idx].copy()
    X_val   = X.iloc[val_idx].copy()
    y_train = y.iloc[train_idx].copy()
    y_val   = y.iloc[val_idx].copy()
"""

In [ ]:
"""from sklearn.cluster import MiniBatchKMeans
from sklearn.model_selection import StratifiedShuffleSplit

df = clean_df.copy().sort_values(by=["R", "Y", "B"]).reset_index(drop=True)

X = df[["R", "Y", "B"]]
y = df[[f"{ch}" for ch in channel_cols]]

# 1. Balanced spatial clusters
kmeans = MiniBatchKMeans(
    n_clusters=8,
    batch_size=32,
    random_state=random_seed,
    n_init="auto"
)
clusters = kmeans.fit_predict(X)

# 2. Ensure each cluster has at least 2 samples
cluster_counts = pd.Series(clusters).value_counts()
valid_clusters = cluster_counts[cluster_counts >= 2].index
clusters = pd.Series(clusters)
mask = clusters.isin(valid_clusters)

X = X[mask]
y = y[mask]
clusters = clusters[mask]

# 3. Stratified split
splitter = StratifiedShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=random_seed
)

train_idx, val_idx = next(splitter.split(X, clusters))

X_train = X.iloc[train_idx].copy()
X_val   = X.iloc[val_idx].copy()
y_train = y.iloc[train_idx].copy()
y_val   = y.iloc[val_idx].copy()
"""

In [ ]:
"""from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import StratifiedShuffleSplit
import numpy as np

df = clean_df.copy().sort_values(by=["R", "Y", "B"]).reset_index(drop=True)

X = df[["R", "Y", "B"]]
y = df[[f"{ch}" for ch in channel_cols]]

# 1. Coarse spatial clustering
kmeans = KMeans(n_clusters=6, random_state=random_seed)
clusters = kmeans.fit_predict(X)

# 2. Stratified split on coarse clusters
splitter = StratifiedShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=random_seed
)

train_idx, val_idx = next(splitter.split(X, clusters))

X_train = X.iloc[train_idx].copy()
X_val   = X.iloc[val_idx].copy()
y_train = y.iloc[train_idx].copy()
y_val   = y.iloc[val_idx].copy()

# 3. Ensure geometric proximity
nbrs = NearestNeighbors(n_neighbors=1).fit(X_train)
distances, _ = nbrs.kneighbors(X_val)

# 4. If any val point is too far, move it to train
threshold = np.percentile(distances, 90)  # adaptive threshold
bad_val = np.where(distances.flatten() > threshold)[0]

if len(bad_val) > 0:
    # move bad val points to train
    move_idx = val_idx[bad_val]
    keep_idx = np.setdiff1d(val_idx, move_idx)

    train_idx = np.concatenate([train_idx, move_idx])
    val_idx = keep_idx

# 5. Final DataFrames
X_train = X.iloc[train_idx].copy()
X_val   = X.iloc[val_idx].copy()
y_train = y.iloc[train_idx].copy()
y_val   = y.iloc[val_idx].copy()
"""

In [ ]:
"""import numpy as np
from sklearn.metrics import pairwise_distances

df = clean_df.copy().sort_values(by=["R", "Y", "B"]).reset_index(drop=True)

X = df[["R", "Y", "B"]].to_numpy()
y = df[[f"{ch}" for ch in channel_cols]]

n_val = int(0.2 * len(X))

# 1. Start with a random point
np.random.seed(123)
val_idx = [np.random.randint(len(X))]

# 2. Precompute distances
D = pairwise_distances(X, X)

# 3. Greedy farthest‑point sampling
while len(val_idx) < n_val:
    # distance to nearest selected val point
    dist_to_val = D[:, val_idx].min(axis=1)
    # pick the farthest point
    next_idx = np.argmax(dist_to_val)
    val_idx.append(next_idx)

val_idx = np.array(val_idx)
train_idx = np.setdiff1d(np.arange(len(X)), val_idx)

# 4. Build DataFrames
X_train = df.iloc[train_idx][["R", "Y", "B"]].copy()
X_val   = df.iloc[val_idx][["R", "Y", "B"]].copy()
y_train = y.iloc[train_idx].copy()
y_val   = y.iloc[val_idx].copy()
"""

In [ ]:
from sklearn.model_selection import GroupKFold

df = clean_df.copy()

# Create pseudo-groups by rounding RYB to nearest 0.1
df["group"] = (
    (df["R"]*10).round().astype(int).astype(str) + "_" +
    (df["Y"]*10).round().astype(int).astype(str) + "_" +
    (df["B"]*10).round().astype(int).astype(str)
)

X = df[["R", "Y", "B"]]
y = df[[f"{ch}" for ch in channel_cols]]
groups = df["group"]

gkf = GroupKFold(n_splits=5)

train_idx, val_idx = next(gkf.split(X, y, groups))

X_train = X.iloc[train_idx].copy()
X_val   = X.iloc[val_idx].copy()
y_train = y.iloc[train_idx].copy()
y_val   = y.iloc[val_idx].copy()


In [ ]:
plt.scatter(X_train["R"], X_train["Y"], c="blue", alpha=0.5, label="train")
plt.scatter(X_val["R"],   X_val["Y"],   c="red",  alpha=0.8, label="val")
plt.legend()
plt.title("Balanced Cluster Train/Val Split")
plt.show()


In [ ]:
#ternary plot 
fig = px.scatter_ternary(
    X_train, 
    a="R", 
    b="Y", 
    c="B",
)

fig.update_traces(marker=dict(color='black', size=5))

fig.update_layout(
    ternary={
        'sum': 100,
        "aaxis": {
            "title": {"text": "Red", "font": {"color": "red", "size": 20}},
            "tickfont": {"color": "red"},
            "linecolor": "red"
        },
        "baxis": {
            "title": {"text": "Yellow", "font": {"color": "yellow", "size": 20}},
            "tickfont": {"color": "black"},
            "linecolor": "yellow"
        },
        "caxis": {
            "title": {"text": "Blue", "font": {"color": "blue", "size": 20}},
            "tickfont": {"color": "blue"},
            "linecolor": "blue"
        }
    }
)
fig.show()

We see a good coverage of our search space from out train data. This should suffice as input for our surrogate model.

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(8,8))
axes = axes.flatten()

for ax, ch in zip(axes, channel_cols):
    sns.histplot(df[ch], kde=True, ax=ax)
    ax.set_title(f"Distribution of {ch}")

plt.tight_layout()
plt.show()


In [ ]:
subset = [f"{ch}" for ch in channel_cols]

sns.pairplot(df[subset], corner=True, diag_kind="kde")
plt.suptitle("Pairwise Relationships Between Channels", y=1.02)
plt.show()


Strong positive correlations: Several channel pairs form clear upward‑sloping clusters, indicating a proportional relationship between channel intensities. 

Smooth, continuous structure: Smooth scatterplots are ideal for Gaussian Process Regression with no extreme outliers.

In [ ]:
plt.figure(figsize=(6, 4))
corr = df[[f"{ch}" for ch in channel_cols]].corr()

sns.heatmap(corr, cmap="coolwarm", annot=False)
plt.title("Correlation Heatmap of Channels")
plt.tight_layout()
plt.show()


### Surrogate Model

In [ ]:
# Separate scalers
x_scaler = StandardScaler()
y_scaler = StandardScaler()

# Fit on training data ONLY
X_train_scaled = x_scaler.fit_transform(X_train)
X_val_scaled  = x_scaler.transform(X_val)

y_train_scaled = y_scaler.fit_transform(y_train)
y_val_scaled  = y_scaler.transform(y_val)


Normalization of the outputs is important as the eight optical channels measured by the OT‑2 span very different numerical ranges.  
For example, `ch410` may produce values in the hundreds, while `ch620` or `ch670` can reach values above 9000. Normalization prevents dominance of high-magnitude channels and enables stability of the surrogate model. It is important to note that the target response must also be normalzied with the same training statistics for proper comparison. 

In [ ]:
"""
channel_cols = [c for c in df.columns if c.startswith("ch")]

# Compute normalization statistics
channel_means = df[channel_cols].mean()
channel_stds = df[channel_cols].std(ddof=0).replace(0, 1.0)

# Apply normalization
for ch in channel_cols:
    df[f"{ch}_norm"] = (df[ch] - channel_means[ch]) / channel_stds[ch]

# Normalize target response
target_norm = (target - channel_means) / channel_stds

print("Normalized target response:")
target_norm
"""



We will primarily use Gaussian Process Regression (GPR) as the surrogate model due to its suitability for small datasets and its ability to provide calibrated uncertainty estimates, which are crucial for active learning.

Using this surrogate, we will implement a sequential acquisition policy (e.g., Expected Improvement) to select up to 26 new RYB combinations to query on the OT-2. At each iteration, the surrogate will be updated with the new data, and we will track:

- Convergence of the predicted response to the target response.
- Improvement in predictive accuracy of the surrogate model.
- Evolution of predictive uncertainty and sample selection in the RYB space.

The following sections describe the data preparation, surrogate modeling, and active learning strategy in detail.


In [ ]:
results = pd.DataFrame(columns=['mae_train', 
                                'rmse_train', 
                                #'maxe_train', 
                                #'mape_train', 
                                'r2_train', 
                                'mae_val', 
                                'rmse_val', 
                                #'maxe_val', 
                                #'mape_val', 
                                'r2_val'])

def compute_metrics(y_true, y_pred, model, set_type):
    mae = mean_absolute_error(y_true, y_pred)
    #mse = mean_squared_error(y_true, y_pred)
    rmse = root_mean_squared_error(y_true, y_pred)
    #maxe = max_error(y_true, y_pred)
    #mape = mean_absolute_percentage_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    metrics_dict = {
    f"mae_{set_type}":  [mae],
    f"rmse_{set_type}":  [rmse],
   # f"maxe_{set_type}": [maxe],
   # f"mape_{set_type}": [mape],
    f"r2_{set_type}":   [r2]}

    for col, val in metrics_dict.items():
            results.loc[model, col] = val

    return results.loc[model]

In [ ]:
kernel = RBF()
gpr_model = GPR(kernel=kernel, random_state=random_seed)
gpr_model.fit(X_train_scaled, y_train_scaled)

In [ ]:
#add row to results with RF_opt
results.loc['RBF'] = [0] * len(results.columns)
y_pred_train_RBF = gpr_model.predict(X_train_scaled)
y_pred_val_RBF = gpr_model.predict(X_val_scaled)

results.loc['RBF'] = compute_metrics(y_train_scaled, y_pred_train_RBF, "RBF", "train")
results.loc['RBF'] = compute_metrics(y_val_scaled, y_pred_val_RBF, "RBF", "val")
results

Radial basis function on its own works well on the trained data but poorly on the validation set.

Let's try adding white kernel to see what happens

In [ ]:
kernel = RBF(length_scale=1.0) + WhiteKernel(noise_level=1.0) 
gpr_model = GPR(kernel=kernel, random_state=random_seed)
gpr_model.fit(X_train_scaled, y_train_scaled)

In [ ]:
#add row to results 
results.loc['RBF_WK'] = [0] * len(results.columns)
y_pred_train_RBF_WK = gpr_model.predict(X_train_scaled)
y_pred_val_RBF_WK = gpr_model.predict(X_val_scaled)

results.loc['RBF_WK'] = compute_metrics(y_train_scaled, y_pred_train_RBF_WK, "RBF_WK", "train")
results.loc['RBF_WK'] = compute_metrics(y_val_scaled, y_pred_val_RBF_WK, "RBF_WK", "val")
results

Does not like white kernel, let's try matern

In [ ]:
kernel = RBF(length_scale=1.0) + Matern(length_scale=1.0, nu=1.5)
gpr_model = GPR(kernel=kernel, random_state=random_seed)
gpr_model.fit(X_train_scaled, y_train_scaled)

In [ ]:
#add row to results 
results.loc['RBF_MT'] = [0] * len(results.columns)
y_pred_train_RBF_MT = gpr_model.predict(X_train_scaled)
y_pred_val_RBF_MT = gpr_model.predict(X_val_scaled)

results.loc['RBF_MT'] = compute_metrics(y_train_scaled, y_pred_train_RBF_MT, "RBF_MT", "train")
results.loc['RBF_MT'] = compute_metrics(y_val_scaled, y_pred_val_RBF_MT, "RBF_MT", "val")
results

Matern is showing severe overfitting, let's try rational quadratic

In [ ]:
kernel = RBF(length_scale=1.0) + RationalQuadratic(length_scale=1.0, alpha=1.0)
gpr_model = GPR(kernel=kernel, random_state=random_seed)
gpr_model.fit(X_train_scaled, y_train_scaled)

In [ ]:
#add row to results 
results.loc['RBF_RQ'] = [0] * len(results.columns)
y_pred_train_RBF_RQ = gpr_model.predict(X_train_scaled)
y_pred_val_RBF_RQ = gpr_model.predict(X_val_scaled)

results.loc['RBF_RQ'] = compute_metrics(y_train_scaled, y_pred_train_RBF_RQ, "RBF_RQ", "train")
results.loc['RBF_RQ'] = compute_metrics(y_val_scaled, y_pred_val_RBF_RQ, "RBF_RQ", "val")
results

Better but just about as good as RBF on its own.

In [ ]:
kernel = RBF(length_scale=1.0) + ConstantKernel(constant_value=1.0)
gpr_model = GPR(kernel=kernel, random_state=random_seed)
gpr_model.fit(X_train_scaled, y_train_scaled)

In [ ]:
#add row to results 
results.loc['RBF_CK'] = [0] * len(results.columns)
y_pred_train_RBF_CK = gpr_model.predict(X_train_scaled)
y_pred_val_RBF_CK = gpr_model.predict(X_val_scaled)

results.loc['RBF_CK'] = compute_metrics(y_train_scaled, y_pred_train_RBF_CK, "RBF_CK", "train")
results.loc['RBF_CK'] = compute_metrics(y_val_scaled, y_pred_val_RBF_CK, "RBF_CK", "val")
results

In [ ]:
kernel = 1.0 * RBF(length_scale=np.ones(3), length_scale_bounds=(1e-2, 1e3)) + WhiteKernel(noise_level=1.0, noise_level_bounds=(1e-3, 1e2))
gpr_model = GPR(kernel=kernel, random_state=random_seed)
gpr_model.fit(X_train_scaled, y_train_scaled)

In [ ]:
#add row to results 
results.loc['RBF_WK2'] = [0] * len(results.columns)
y_pred_train_RBF_WK2 = gpr_model.predict(X_train_scaled)
y_pred_val_RBF_WK2 = gpr_model.predict(X_val_scaled)

results.loc['RBF_WK2'] = compute_metrics(y_train_scaled, y_pred_train_RBF_WK2, "RBF_WK2", "train")
results.loc['RBF_WK2'] = compute_metrics(y_val_scaled, y_pred_val_RBF_WK2, "RBF_WK2", "val")
results

In [ ]:
kernel = RBF(length_scale=[1,1,1], length_scale_bounds=(1e-2, 1e3)) #+ WhiteKernel(noise_level=1.0, noise_level_bounds=(1e-3, 1e2))
gpr = GPR(kernel=kernel, normalize_y=False,random_state=random_seed)
multi_gpr = MultiOutputRegressor(gpr)
multi_gpr.fit(X_train_scaled, y_train_scaled)

In [ ]:
#add row to results 
results.loc['MOR'] = [0] * len(results.columns)
y_pred_train_MOR = multi_gpr.predict(X_train_scaled)
y_pred_val_MOR = multi_gpr.predict(X_val_scaled)

results.loc['MOR'] = compute_metrics(y_train_scaled, y_pred_train_MOR, "MOR", "train")
results.loc['MOR'] = compute_metrics(y_val_scaled, y_pred_val_MOR, "MOR", "val")
results

In [ ]:
kernel = RBF(length_scale=[1,1,1], length_scale_bounds=(1e-2, 1e3)) + WhiteKernel(noise_level=1.0, noise_level_bounds=(1e-3, 1e2))
gpr = GPR(kernel=kernel, normalize_y=False,random_state=random_seed)
multi_gpr = MultiOutputRegressor(gpr)
multi_gpr.fit(X_train_scaled, y_train_scaled)

In [ ]:
#add row to results 
results.loc['MOR_WK'] = [0] * len(results.columns)
y_pred_train_MOR_WK = multi_gpr.predict(X_train_scaled)
y_pred_val_MOR_WK = multi_gpr.predict(X_val_scaled)

results.loc['MOR_WK'] = compute_metrics(y_train_scaled, y_pred_train_MOR_WK, "MOR_WK", "train")
results.loc['MOR_WK'] = compute_metrics(y_val_scaled, y_pred_val_MOR_WK, "MOR_WK", "val")
results

In [ ]:
"""class MultitaskGPModel(gpytorch.models.ApproximateGP):
    def __init__(self, num_tasks, input_dim):
        # Variational distribution
        inducing_points = torch.randn(64, input_dim)
        variational_distribution = gpytorch.variational.MultitaskVariationalDistribution(
            inducing_points.size(0), num_tasks=num_tasks
        )
        variational_strategy = gpytorch.variational.MultitaskVariationalStrategy(
            gpytorch.variational.VariationalStrategy(
                self, inducing_points, variational_distribution, learn_inducing_locations=True
            ),
            num_tasks=num_tasks
        )

        super().__init__(variational_strategy)

        # Shared kernel across tasks
        self.mean_module = ConstantMean()
        self.covar_module = ScaleKernel(RBFKernel(ard_num_dims=input_dim))

        # Task covariance (ICM)
        self.task_covar_module = IndexKernel(num_tasks=num_tasks)

    def forward(self, x):
        mean_x = self.mean_module(x)
        covar_x = self.covar_module(x)
        task_covar = self.task_covar_module.covar_matrix
        return MultitaskMultivariateNormal.from_batch_mvn(
            gpytorch.distributions.MultivariateNormal(mean_x, covar_x),
            task_covar
        )
"""

In [ ]:
"""class MultitaskGPModel(gpytorch.models.ExactGP):
    def __init__(self, train_x, train_y, likelihood, num_tasks):
        super().__init__(train_x, train_y, likelihood)
        
        self.mean_module = ConstantMean()
        
        # Shared RBF kernel across tasks
        base_kernel = ScaleKernel(RBFKernel(ard_num_dims=train_x.shape[-1]))
        
        # ICM multitask kernel
        self.covar_module = MultitaskKernel(
            base_kernel=base_kernel,
            num_tasks=num_tasks,
            rank=1
        )

    def forward(self, x):
        mean_x = self.mean_module(x)
        covar_x = self.covar_module(x)
        return gpytorch.distributions.MultitaskMultivariateNormal(mean_x, covar_x)
"""

In [ ]:
"""class MultitaskGPModel(gpytorch.models.ExactGP):
    def __init__(self, train_x, train_y, likelihood, num_tasks):
        super().__init__(train_x, train_y, likelihood)

        self.mean_module = ConstantMean()

        # Base kernel (shared across tasks)
        data_kernel = ScaleKernel(RBFKernel(ard_num_dims=train_x.shape[-1]))

        # Multitask kernel using ICM
        self.covar_module = MultitaskKernel(
            data_covar_module=data_kernel,
            num_tasks=num_tasks,
            rank=1
        )

    def forward(self, x):
        mean_x = self.mean_module(x)
        covar_x = self.covar_module(x)
        return gpytorch.distributions.MultitaskMultivariateNormal(mean_x, covar_x)
"""

In [ ]:
class MultitaskGPModel(gpytorch.models.ExactGP):
    def __init__(self, train_x, train_y, likelihood, num_tasks):
        super().__init__(train_x, train_y, likelihood)

        # Multitask mean: wraps a base mean and expands to (N, num_tasks)
        self.mean_module = MultitaskMean(
            base_means=ConstantMean(),
            num_tasks=num_tasks
        )

        # Base kernel over inputs
        data_kernel = ScaleKernel(RBFKernel(ard_num_dims=train_x.shape[-1]))

        # ICM multitask kernel
        self.covar_module = MultitaskKernel(
            data_covar_module=data_kernel,
            num_tasks=num_tasks,
            rank=1
        )

    def forward(self, x):
        mean_x = self.mean_module(x)      # shape: (N, num_tasks)
        covar_x = self.covar_module(x)    # multitask covariance
        return gpytorch.distributions.MultitaskMultivariateNormal(mean_x, covar_x)


In [ ]:
X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_t = torch.tensor(y_train_scaled, dtype=torch.float32)

num_tasks = y_train_t.shape[1]


In [ ]:
X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_t = torch.tensor(y_train_scaled, dtype=torch.float32)
num_tasks = y_train_t.shape[1]

likelihood = gpytorch.likelihoods.MultitaskGaussianLikelihood(num_tasks=num_tasks)
model = MultitaskGPModel(X_train_t, y_train_t, likelihood, num_tasks)

model.train()
likelihood.train()

optimizer = torch.optim.Adam(model.parameters(), lr=0.05)
mll = gpytorch.mlls.ExactMarginalLogLikelihood(likelihood, model)

for i in range(300):
    optimizer.zero_grad()
    output = model(X_train_t)
    loss = -mll(output, y_train_t)
    loss.backward()
    optimizer.step()


In [ ]:
model.eval()
likelihood.eval()

X_test_t = torch.tensor(X_val_scaled, dtype=torch.float32)

with torch.no_grad():
    preds_train = likelihood(model(X_train_t)) 
    preds_val = likelihood(model(X_test_t)) 

y_pred_train_gpy = preds_train.mean.numpy() 
y_pred_val_gpy = preds_val.mean.numpy()


In [ ]:
results.loc['GPyTorch'] = [0] * len(results.columns)

results.loc['GPyTorch'] = compute_metrics(
    y_train_scaled, 
    y_pred_train_gpy, 
    "GPyTorch", 
    "train"
)

results.loc['GPyTorch'] = compute_metrics(
    y_val_scaled, 
    y_pred_val_gpy, 
    "GPyTorch", 
    "val"
)


In [ ]:
results

# Fix saved results

In [ ]:
"""#save results table to csv with name simple split
results.to_csv("simple_split_results.csv", index=True)"""
#read results table from csv
#simple_split = pd.read_csv("simple_split_results.csv", index_col=0)
#simple_split

In [ ]:
X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_t = torch.tensor(y_train_scaled, dtype=torch.float32)

num_tasks = y_train_t.shape[1]

likelihood = gpytorch.likelihoods.MultitaskGaussianLikelihood(num_tasks=num_tasks)
model = MultitaskGPModel(X_train_t, y_train_t, likelihood, num_tasks)

model.train()
likelihood.train()

# Improvement 1: Higher LR
optimizer = torch.optim.Adam(model.parameters(), lr=0.1)

# Improvement 2: More iterations
num_iters = 1000

# Improvement 3: Jitter for stability
mll = gpytorch.mlls.ExactMarginalLogLikelihood(likelihood, model)

for i in range(num_iters):
    optimizer.zero_grad()

    with gpytorch.settings.cholesky_jitter(1e-3):
        output = model(X_train_t)
        loss = -mll(output, y_train_t)

    loss.backward()
    optimizer.step()

    if i % 100 == 0:
        print(f"Iter {i} - Loss: {loss.item():.4f}")


In [ ]:
model.eval()
likelihood.eval()

X_test_t = torch.tensor(X_val_scaled, dtype=torch.float32)

with torch.no_grad():
    preds_train = likelihood(model(X_train_t))
    preds_val   = likelihood(model(X_test_t))
    

y_pred_train_gpy_mod = preds_train.mean.numpy()
y_pred_val_gpy_mod   = preds_val.mean.numpy()



In [ ]:
results.loc['GPyTorch_mod'] = [0] * len(results.columns)

results.loc['GPyTorch_mod'] = compute_metrics(
    y_train_scaled,
    y_pred_train_gpy_mod,
    "GPyTorch",
    "train"
)

results.loc['GPyTorch_mod'] = compute_metrics(
    y_val_scaled,
    y_pred_val_gpy_mod,
    "GPyTorch",
    "val"
)


In [ ]:
results

In [ ]:
X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
X_val_t   = torch.tensor(X_val_scaled,   dtype=torch.float32)
y_train_t = torch.tensor(y_train_scaled, dtype=torch.float32)

num_tasks = y_train_t.shape[1]
input_dim = X_train_t.shape[-1]

# ----------------------------
class MultitaskSMGPModel(gpytorch.models.ExactGP):
    def __init__(self, train_x, train_y, likelihood, num_tasks, input_dim):
        super().__init__(train_x, train_y, likelihood)

        self.mean_module = MultitaskMean(
            base_means=ConstantMean(),
            num_tasks=num_tasks
        )

        sm_kernel = SpectralMixtureKernel(
            num_mixtures=4,
            ard_num_dims=input_dim
        )

        self.covar_module = MultitaskKernel(
            data_covar_module=sm_kernel,
            num_tasks=num_tasks,
            rank=1
        )

    def forward(self, x):
        mean_x = self.mean_module(x)
        covar_x = self.covar_module(x)
        return gpytorch.distributions.MultitaskMultivariateNormal(mean_x, covar_x)

likelihood = gpytorch.likelihoods.MultitaskGaussianLikelihood(num_tasks=num_tasks)
model = MultitaskSMGPModel(X_train_t, y_train_t, likelihood, num_tasks, input_dim)


model.train()
likelihood.train()

optimizer = torch.optim.Adam(model.parameters(), lr=0.05)
mll = gpytorch.mlls.ExactMarginalLogLikelihood(likelihood, model)

num_iters = 800
for i in range(num_iters):
    optimizer.zero_grad()
    with gpytorch.settings.cholesky_jitter(1e-3):
        output = model(X_train_t)
        loss = -mll(output, y_train_t)
    loss.backward()
    optimizer.step()
    if i % 100 == 0:
        print(f"Iter {i} - Loss: {loss.item():.4f}")

###
model.eval()
likelihood.eval()

with torch.no_grad(), gpytorch.settings.cholesky_jitter(1e-3):
    preds_train = likelihood(model(X_train_t))
    preds_val   = likelihood(model(X_val_t))

y_pred_train_gpy_mod2 = preds_train.mean.numpy()
y_pred_val_gpy_mod2   = preds_val.mean.numpy()

# (optional) back to original units
"""y_train_pred = y_scaler.inverse_transform(y_pred_train_gpy_mod2)
y_val_pred   = y_scaler.inverse_transform(y_pred_val_gpy_mod2)
y_train_true = y_train.values
y_val_true   = y_val.values
"""

# use original units for interpretability
"""compute_metrics(y_train_true, y_train_pred, "GPyTorch_SM", "train")
compute_metrics(y_val_true,   y_val_pred,   "GPyTorch_SM", "val")

print(results)"""


In [ ]:
results.loc['GPyTorch_mod2'] = [0] * len(results.columns)

results.loc['GPyTorch_mod2'] = compute_metrics(
    y_train_scaled,
    y_pred_train_gpy_mod2,
    "GPyTorch_mod2",
    "train"
)

results.loc['GPyTorch_mod2'] = compute_metrics(
    y_val_scaled,
    y_pred_val_gpy_mod2,
    "GPyTorch_mod2",
    "val"
)

results


make parity of train and test set

### Acquisition Policy

In [ ]:
import numpy as np
import torch gp_predict_mean
from scipy.stats import norm

def_std(model, likelihood, X_scaled):
    likelihood.eval.tensor(X_scaled model.eval()
   ()

    X_t = torch, dtype=torch.float32)

    with torch preds = likelihood.no_grad():
       (model(X_t))
        mean = preds.mean.numpy()
        var  = preds.variance.numpy()
        std  = np.sqrt(var)

    return mean, std


In [ ]:
def acquisition_EI(model, likelihood, X_scaled, best = objective(mean_y):
    mean, std = gp_predict_mean_std(model, likelihood, X_scaled)
    mu)
    sigma = objective(std)

    sigma = np.maximum(sigma, 1e-9)
    Z = (mu - best_y) / sigma

    EI = (mu - best_y) * norm.cdf(Z) + sigma * norm.pdf(Z)
    EI[sigma < 1e-9] = 0.0

    return EI
